# Evaluation on the train set

Checks for each stage of the pipeline, scored against `train_labels.csv`:

1. **Clustering ceiling**: the best score a scene clustering can reach if every clustered image gets a perfect pose. A clustering fixed *before* matching caps the final score at this value.
2. **Pair shortlist**: whether the candidate pairs can still reconstruct each scene, and how many pairs need matching.
3. **Full submission**: the official IMC2025 metric (`imc/metric.py`) on a submission csv.

Run from the repository root. Data is read from `image-matching-challenge-2025/` or from the folder in the `IMC_DATA` environment variable.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.cluster import HDBSCAN

from imc.data import DATA_ROOT, list_images, read_scenes
from imc.evaluate import clustering_ceiling, score_submission, shortlist_report
from imc.retrieval import embed_images, load_dinov2, pick_device, shortlist_pairs

images = list_images('train')
scenes = read_scenes(DATA_ROOT / 'train_labels.csv')
device = pick_device()
print(device, {dataset: len(paths) for dataset, paths in images.items()})

## Global descriptors

DINOv2 class token (`cls`) and max-pooled patch tokens (`mac`), with images resized keeping their aspect ratio. Worth trying `dinov2_vitb14` / `dinov2_vitl14` too.

In [ ]:
model = load_dinov2('dinov2_vits14', device=device)
desc = {dataset: embed_images(paths, model) for dataset, paths in images.items()}

## 1. Clustering ceiling

The historical HDBSCAN clustering baseline, scored as if every clustered image got a perfect pose. `one_cluster` is the same ceiling with every image of the dataset in a single cluster; the true scenes would score 100.

In [ ]:
def hdbscan_clusters(X, paths, **kwargs):
    labels = HDBSCAN(**kwargs).fit_predict(X)
    clusters = {}
    for path, label in zip(paths, labels):
        clusters.setdefault('outliers' if label == -1 else f'cluster{label}', []).append(Path(path).name)
    return clusters


ceiling = {}
for dataset, paths in images.items():
    clusters = hdbscan_clusters(desc[dataset]['cls'], paths, min_cluster_size=3, copy=True)
    ceiling[dataset] = {
        **clustering_ceiling(clusters, scenes[dataset]),
        'clusters': sum(name != 'outliers' for name in clusters),
        'one_cluster': clustering_ceiling({'cluster0': [p.name for p in paths]}, scenes[dataset])['score'],
    }
ceiling = pd.DataFrame(ceiling).T
ceiling.loc['mean'] = ceiling.mean()
ceiling.round(1)

## 2. Pair shortlist

Candidate pairs before any matching, per strategy:

- `coverage`: share of a scene in its largest group of linked same-scene candidates (mean over scenes; `worst` is the lowest). No reconstruction can correctly register more of the scene.
- `isolated`: scene images without a single same-scene candidate.
- `cross`: share of candidates joining different scenes or outliers.
- `pairs` / `of_all`: matching cost, total and as a share of exhaustive matching.

`hdbscan groups` pairs images only inside HDBSCAN clusters (the historical baseline): its noise images get no pairs and its fragments of a scene are never linked. `top-k` pairs every image with its k nearest neighbours (`imc.retrieval.shortlist_pairs`).

In [ ]:
def within_cluster_pairs(clusters, paths):
    index = {Path(p).name: i for i, p in enumerate(paths)}
    groups = [sorted(index[name] for name in names) for cluster, names in clusters.items() if cluster != 'outliers']
    return sorted((i, j) for group in groups for a, i in enumerate(group) for j in group[a + 1:])


strategies = {'hdbscan groups': lambda X, paths: within_cluster_pairs(hdbscan_clusters(X, paths, min_cluster_size=3, copy=True), paths)}
for k in (5, 10, 20, 30):
    strategies[f'top-{k}'] = lambda X, paths, k=k: shortlist_pairs(X, k=k)

reports = pd.DataFrame([
    {'descriptor': kind, 'strategy': name, 'dataset': dataset,
     **shortlist_report(make_pairs(desc[dataset][kind], paths), paths, scenes[dataset])}
    for kind in ('cls', 'mac')
    for name, make_pairs in strategies.items()
    for dataset, paths in images.items()
])
reports.groupby(['descriptor', 'strategy'], sort=False).agg(
    pairs=('pairs', 'sum'), of_all=('of_all', 'mean'), cross=('cross', 'mean'),
    coverage=('coverage', 'mean'), worst=('worst', 'min'), isolated=('isolated', 'sum'),
).round(3)

In [ ]:
reports[reports['descriptor'] == 'cls'].pivot(index='dataset', columns='strategy', values='coverage')[list(strategies)].round(2)

## 3. Full submission

Once the pipeline produces poses, write one row per train image and score it with the official metric:

```python
from imc.evaluate import write_submission

# cluster: 'cluster0', 'cluster1', ... or 'outliers'; R (3x3), t (3): cam_from_world, None if unregistered
rows = [(dataset, cluster, image, R, t), ...]
write_submission('outputs/train_submission.csv', rows)
score_submission('outputs/train_submission.csv', DATA_ROOT / 'train_labels.csv', DATA_ROOT / 'train_thresholds.csv')
```

or from the command line: `python -m imc.evaluate outputs/train_submission.csv`. Only the datasets in the file are scored. The metric aligns cameras over every triplet of images, so scoring all of train takes a while (over 10 minutes); score a few datasets while iterating.

Sanity check of the scoring loop: the ground truth itself has to score 100.

In [ ]:
Path('outputs').mkdir(exist_ok=True)
labels = pd.read_csv(DATA_ROOT / 'train_labels.csv')
labels[labels['dataset'].isin(['ETs', 'imc2023_haiper'])].to_csv('outputs/gt_as_submission.csv', index=False)
score_submission('outputs/gt_as_submission.csv', DATA_ROOT / 'train_labels.csv', DATA_ROOT / 'train_thresholds.csv')